In [ ]:
# Codice per estrazione degli abstract da arXiv pubblicati prima del 2022 (pre GPT)

!pip install -q arxiv pandas tqdm

import arxiv
import pandas as pd
import re
from tqdm.notebook import tqdm

# Pulizia preliminare del testo grezzo restituito dalle API di arXiv
def clean_abstract(text):
    # normalizza ritorni a capo e spazi multipli in un singolo spazio
    text = re.sub(r'\s+', ' ', text)
    # rimuove eventuali comandi LaTeX di citazione esplicita tipo \cite{...}
    text = re.sub(r'\\cite\{.*?\}', '', text)
    return text.strip()

# 5 sottocategorie di Computer Science per avere varieta'
# LG (Machine Learning), CV (Vision), SE (Software Eng), CR (Security), DB (Databases)
categories = ["cs.LG", "cs.CV", "cs.SE", "cs.CR", "cs.DB"]
papers_per_category = 300 # 300 x 5 = 1500 abstract human

# Configurazione client con delay di 3s per rispettare i rate limit di arXiv
client = arxiv.Client(
    page_size=100,
    delay_seconds=3.0,
    num_retries=3
)

human_data = []
seen_ids = set() # evita duplicati in caso di paper postati su piu categorie
print("Inizio download degli abstract human da arXiv (Periodo 2018-2021)...")

for cat in categories:
    # Filtro temporale 2018-2021 (pre GPT) per avere certezza di testi scritti solo da umani
    query = f"cat:{cat} AND submittedDate:[201801010000 TO 202112312359]"

    # Margine extra (+150) per compensare i paper scartati dal filtro sulla lunghezza
    search = arxiv.Search(
        query=query,
        max_results=papers_per_category + 150,
        sort_by=arxiv.SortCriterion.SubmittedDate,
        sort_order=arxiv.SortOrder.Descending
    )

    count = 0
    for result in tqdm(client.results(search), total=papers_per_category, desc=f"Scaricando {cat}"):
        if count >= papers_per_category:
            break

        # Estrae l'ID base rimuovendo il suffisso di versione (es. v1, v2)
        paper_id = result.get_short_id().split('v')[0]
        if paper_id in seen_ids:
            continue

        cleaned_text = clean_abstract(result.summary)
        cleaned_title = re.sub(r'\s+', ' ', result.title).strip()
        word_count = len(cleaned_text.split())

        # Filtra solo abstract di lunghezza standard (tra 80 e 300 parole)
        if 80 <= word_count <= 300:
            seen_ids.add(paper_id)
            human_data.append({
                "pair_id": f"arxiv_{paper_id}",
                "category": cat,
                "title": cleaned_title,
                "text": cleaned_text,
                "label": 0,              # 0 = Human
                "gen_type": "human",
                "source_model": "human",
                "word_count": word_count # lunghezza di riferimento per la generazione AI successiva
            })
            count += 1

# Shuffle globale (seed fisso per riproducibilita) per mescolare le categorie
# prima di assegnare i due task di generazione AI
df_human = pd.DataFrame(human_data)
df_human = df_human.sample(frac=1, random_state=42).reset_index(drop=True)

# Split 50/50 dei task AI 750 generati da titolo, 750 parafrasati da originale
df_human['target_ai_task'] = ['from_title' if i < 750 else 'paraphrased' for i in range(len(df_human))]

output_file = "step1A_human_abstracts.csv"
df_human.to_csv(output_file, index=False)

print(f"\nCompletato! Salvate {len(df_human)} righe in '{output_file}'.")
display(df_human[['pair_id', 'category', 'title', 'word_count', 'target_ai_task']].head(3))

In [ ]:
# Codice di generazione degli abstact AI usando AWS bedrock e unione finale di entrambi

!pip install -q boto3 pandas tqdm

import boto3
import pandas as pd
import re
import time
import os
from tqdm.notebook import tqdm
from botocore.exceptions import ClientError

# Credenziali AWS per Bedrock (DA RIMUOVERE PRIMA DI PUBBLICAZIONE)
AWS_ACCESS_KEY = "YOUR_AWS_ACCESS_KEY"
AWS_SECRET_KEY = "YOUR_AWS_SECRET_KEY"
AWS_REGION = "us-east-1"

# Modello utilizzato su Bedrock Llama 3 70B Instruct
MODEL_ID = "meta.llama3-70b-instruct-v1:0"
MODEL_NAME_LABEL = "llama-3-70b"

bedrock = boto3.client(
    service_name="bedrock-runtime",
    region_name=AWS_REGION,
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY
)

# Rimozione URL (per evitare data leakage tra human e AI) ed eventuali preamboli del LLM
def clean_generated_text(text):
    text = re.sub(r'http[s]?://\S+', '', text)
    text = re.sub(r'^(Here is|Here\'s|Sure|Certainly|Abstract:).*?:\s*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\s+', ' ', text)
    return text.strip().strip('"')

# Costruzione dinamica dei prompt per i due task (from_title vs paraphrased)
# con vincolo di lunghezza (+/- 15 parole rispetto al testo umano corrispondente)
def build_prompt(row):
    target_words = row['word_count']
    title = row['title']

    system_prompt = (
        "You are an expert computer science researcher writing an abstract for an academic paper. "
        "Output ONLY the raw text of the abstract. Do NOT include any introductory phrases "
        "(like 'Here is the abstract'), titles, markdown formatting, or URLs. "
        "Vary your opening sentences naturally and avoid cliches like 'In this paper, we delve into'."
    )

    if row['target_ai_task'] == 'from_title':
        user_prompt = (
            f"Write a realistic, rigorous scientific abstract for a computer science paper titled: \"{title}\".\n"
            f"The abstract MUST be approximately {target_words} words long (strictly between {target_words - 15} and {target_words + 15} words). "
            f"Include realistic context, methodology, and findings suitable for the '{row['category']}' domain."
        )
    else: # paraphrased
        clean_original = clean_generated_text(row['text'])
        user_prompt = (
            f"Rewrite and polish the following scientific abstract to improve its academic tone, flow, and clarity. "
            f"Keep all original technical facts, metrics, and meanings intact, but completely rephrase the sentences using your own academic style.\n"
            f"Target length: approximately {target_words} words (strictly between {target_words - 15} and {target_words + 15} words).\n\n"
            f"Original abstract:\n{clean_original}"
        )

    return system_prompt, user_prompt

# Caricamento abstract human e allineamento pulizia URL e conteggio parole
df_human = pd.read_csv("step1A_human_abstracts.csv")
df_human['text'] = df_human['text'].apply(clean_generated_text)
df_human['word_count'] = df_human['text'].apply(lambda x: len(x.split()))

# Gestione checkpoint su file per riprendere in caso di disconnessione di Colab
checkpoint_file = "step1B_ai_checkpoint.csv"
if os.path.exists(checkpoint_file):
    df_ai_existing = pd.read_csv(checkpoint_file)
    completed_ids = set(df_ai_existing['pair_id'].tolist())
    ai_rows = df_ai_existing.to_dict('records')
    print(f"Trovato checkpoint precedente: {len(completed_ids)} abstract gia' generati. Riprendo...")
else:
    completed_ids = set()
    ai_rows = []

print(f"Avvio generazione di {len(df_human) - len(completed_ids)} abstract AI tramite {MODEL_ID}...")

for idx, row in tqdm(df_human.iterrows(), total=len(df_human), desc="Generazione AWS Bedrock"):
    if row['pair_id'] in completed_ids:
        continue

    sys_prompt, usr_prompt = build_prompt(row)

    # Gestione retry con backoff in caso di throttling dalle API AWS
    for attempt in range(4):
        try:
            response = bedrock.converse(
                modelId=MODEL_ID,
                system=[{"text": sys_prompt}],
                messages=[{"role": "user", "content": [{"text": usr_prompt}]}],
                inferenceConfig={
                    "maxTokens": 512,
                    "temperature": 0.7, # 0.7 bilanciamento tra varieta lessicale e coerenza tecnica [default per LLama3 in questi casi]
                    "topP": 0.9         # 0.9 soglia di probabilita cumulativa [default per LLama3 in questi casi]
                }
            )

            ai_text = response['output']['message']['content'][0]['text']
            ai_text = clean_generated_text(ai_text)
            ai_word_count = len(ai_text.split())

            ai_rows.append({
                "pair_id": row['pair_id'],
                "category": row['category'],
                "title": row['title'],
                "text": ai_text,
                "label": 1,                          # 1 = AI Generated
                "gen_type": row['target_ai_task'],   # 'from_title' oppure 'paraphrased'
                "source_model": MODEL_NAME_LABEL,
                "word_count": ai_word_count,
                "target_ai_task": row['target_ai_task']
            })
            break

        except ClientError as e:
            print(f"\nAttesa per rate limit AWS (Tentativo {attempt+1}/4): {e}")
            time.sleep(5 * (attempt + 1))

    # Salvataggio intermedio ogni 50 generazioni (utile in caso di disconnessione da Colab)
    if len(ai_rows) % 50 == 0:
        pd.DataFrame(ai_rows).to_csv(checkpoint_file, index=False)

df_ai = pd.DataFrame(ai_rows)
df_ai.to_csv(checkpoint_file, index=False)

# Merge finale unisco i 1500 human e i 1500 AI ordinando per pair_id
# Ogni abstract human (label 0) e' seguito dalla sua controparte AI (label 1)
df_final = pd.concat([df_human, df_ai], ignore_index=True)
df_final = df_final.sort_values(by=['pair_id', 'label']).reset_index(drop=True)
df_final = df_final.drop(columns=['target_ai_task']) # info gia' presente in 'gen_type'

final_filename = "step1_complete_dataset_3000.csv"
df_final.to_csv(final_filename, index=False)

print(f"\nSTEP 1 COMPLETATO! Creato '{final_filename}' con {len(df_final)} righe totali.")
display(df_final.head(6))